In [23]:
# import fastf1
import pandas as pd
import os
import sys
import importlib

sys.path.append(os.path.abspath(".."))

import src.fantasy
import src.models
import src.features

importlib.reload(src.fantasy)
importlib.reload(src.models)
importlib.reload(src.features)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from src.fantasy import build_fantasy_team, generate_explanations
from src.data_loader import save_dataset, load_dataset
from src.features import build_features
from src.models import prepare_data, train_model, predict, build_fantasy_table, evaluate_model, tune_gainer_weights
# fastf1.Cache.enable_cache("../data/cache")

In [24]:
df_2023 = load_dataset("../data/processed/race_results_2023.csv")
df_2024 = load_dataset("../data/processed/race_results_2024.csv")

df = pd.concat([df_2023, df_2024], ignore_index=True)

In [25]:
df_2023 = build_features(df_2023)
df_2024 = build_features(df_2024)

In [26]:
X_train, y_train, X_test, y_test = prepare_data(df_2023, df_2024)

model = train_model(X_train, y_train)
predictions = predict(model,X_test)

fantasy_table = build_fantasy_table(df_2024, predictions)

In [27]:
team = build_fantasy_team(fantasy_table, "Qatar Grand Prix")
team[["Abbreviation", "GridPosition", "Predicted", "PositionChange", "FantasyValue", "PickCategory", "GainerScore"]]


,Abbreviation,GridPosition,Predicted,PositionChange,FantasyValue,PickCategory,GainerScore
459,VER,2.0,1.36,1.0,1.2,Safe,NaN
460,LEC,5.0,3.06,3.0,2.2,Safe,NaN
461,PIA,4.0,3.29,1.0,1.2,Safe,NaN
463,GAS,11.0,9.67,6.0,3.7,Value,0.465
466,ZHO,12.0,10.13,4.0,2.4,Value,0.135
465,ALO,8.0,9.23,1.0,0.9,Risk,-2.261


In [28]:
team = build_fantasy_team(fantasy_table, "Qatar Grand Prix")
team = generate_explanations(team)
team[["Abbreviation", "PickCategory", "Explanation"]]

,Abbreviation,PickCategory,Explanation
459,VER,Safe,"Safe Pick: starts P2, predicted P1, consistent..."
460,LEC,Safe,"Safe Pick: starts P5, predicted P3, consistent..."
461,PIA,Safe,"Safe Pick: starts P4, predicted P3, consistent..."
463,GAS,Value,"Value Pick: starts P11, predicted P10, average..."
466,ZHO,Value,"Value Pick: starts P12, predicted P10, average..."
465,ALO,Risk,"Risk Pick: starts p8, predicted P9, inconsiste..."


In [ ]:
best_weights = tune_gainer_weights(fantasy_table)
best_weights

In [ ]:
print(best_weights)

{'avg_position_change': np.float64(0.1), 'predicted': np.float64(0.1), 'confidence': np.float64(0.1), 'grid_gap': np.float64(0.4)}
